Load all four split datasets

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import numpy as np
import pandas as pd

folder = Path("/content/drive/MyDrive/Project2_SolarPV")
results_folder = folder / "results"
results_folder.mkdir(exist_ok=True)

HORIZONS = {
    1: 15,
    2: 30,
    3: 45,
    4: 60,
}

data = {}

for h in HORIZONS:
    path = folder / f"ds_h{h}_split.csv"

    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run 02b and 03b first.")

    d = pd.read_csv(path, parse_dates=["DATE_TIME", "TARGET_TIME"])
    data[h] = d.sort_values("DATE_TIME").reset_index(drop=True)

    print(
        f"{HORIZONS[h]} minutes: {len(d)} rows; "
        f"{d['SPLIT'].value_counts().to_dict()}"
    )

Mounted at /content/drive
15 minutes: 3111 rows; {'train': 2159, 'val': 479, 'test': 473}
30 minutes: 3101 rows; {'train': 2152, 'val': 478, 'test': 471}
45 minutes: 3092 rows; {'train': 2145, 'val': 477, 'test': 470}
60 minutes: 3084 rows; {'train': 2139, 'val': 476, 'test': 469}


Define the feature sets and candidate models



In [4]:
weather_features = [
    "IRRADIATION",
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
]

time_features = ["hour_sin", "hour_cos"]

power_history_features = [
    "AC_POWER_lag1",
    "AC_POWER_lag2",
    "AC_POWER_lag3",
]

feature_sets = {
    "A_weather": weather_features,
    "A_time": weather_features + time_features,
    "B_history": weather_features + power_history_features,
    "B_time": weather_features + power_history_features + time_features,
}

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import ParameterGrid

parameter_grids = {
    "LinearRegression": [{}],
    "RandomForest": list(ParameterGrid({
        "max_depth": [None, 10],
        "min_samples_leaf": [1, 3],
    })),
    "GradientBoosting": list(ParameterGrid({
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3],
    })),
}

def make_model(model_name, params):
    if model_name == "LinearRegression":
        return make_pipeline(
            StandardScaler(),
            LinearRegression(),
        )

    if model_name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            **params,
        )

    if model_name == "GradientBoosting":
        return GradientBoostingRegressor(
            random_state=42,
            **params,
        )

    raise ValueError(f"Unknown model: {model_name}")

    print("Feature sets and model configurations are ready.")

for name, features in feature_sets.items():
    print(f"{name}: {features}")

print("Models:", list(parameter_grids.keys()))

A_weather: ['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']
A_time: ['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'hour_sin', 'hour_cos']
B_history: ['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'AC_POWER_lag1', 'AC_POWER_lag2', 'AC_POWER_lag3']
B_time: ['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'AC_POWER_lag1', 'AC_POWER_lag2', 'AC_POWER_lag3', 'hour_sin', 'hour_cos']
Models: ['LinearRegression', 'RandomForest', 'GradientBoosting']


##Tune on training data and select using validation daytime RMSE
The test set is not used for tuning or model selection.

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

validation_rows = []

for h, d in data.items():
    train = d[d["SPLIT"] == "train"]
    val = d[d["SPLIT"] == "val"]

    daytime_mask = (
        (val["TARGET_TIME"].dt.hour >= 6)
        & (val["TARGET_TIME"].dt.hour < 18)
    ).to_numpy()

    for feature_name, features in feature_sets.items():
        X_train = train[features]
        y_train = train["TARGET"]

        X_val = val[features]
        y_val = val["TARGET"].to_numpy()

        for model_name, candidates in parameter_grids.items():
            for params in candidates:
                model = make_model(model_name, params)
                model.fit(X_train, y_train)

                # Clip predictions because power cannot be negative.
                pred = np.clip(model.predict(X_val), 0, None)

                validation_rows.append({
                    "horizon_steps": h,
                    "horizon_minutes": HORIZONS[h],
                    "feature_set": feature_name,
                    "model": model_name,
                    "params": params,
                    "validation_MAE_all": mean_absolute_error(y_val, pred),
                    "validation_RMSE_all": mean_squared_error(y_val, pred) ** 0.5,
                    "validation_RMSE_daytime": (
                        mean_squared_error(
                            y_val[daytime_mask],
                            pred[daytime_mask],
                        ) ** 0.5
                    ),
                })

validation_results = pd.DataFrame(validation_rows)

# Pick the best algorithm and settings for each horizon/feature set
# by daytime validation RMSE.
best_by_feature_set = (
    validation_results
    .sort_values("validation_RMSE_daytime")
    .groupby(["horizon_steps", "feature_set"], as_index=False)
    .first()
)

display(best_by_feature_set.round(3))

,horizon_steps,feature_set,horizon_minutes,model,params,validation_MAE_all,validation_RMSE_all,validation_RMSE_daytime
0,1,A_time,15,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'n_est...",976.921,1839.426,2594.194
1,1,A_weather,15,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est...",1051.679,1952.412,2746.460
2,1,B_history,15,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 3}",940.726,1903.213,2682.919
3,1,B_time,15,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'n_est...",985.618,1841.284,2595.451
4,2,A_time,30,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'n_est...",1173.200,2177.561,3061.831
5,2,A_weather,30,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est...",1347.615,2418.839,3389.195
6,2,B_history,30,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 3}",1251.383,2447.206,3441.841
7,2,B_time,30,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'n_est...",1207.050,2204.655,3099.696
8,3,A_time,45,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'n_est...",1260.116,2282.664,3199.982
9,3,A_weather,45,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 3}",1364.157,2582.391,3628.525


Refit selected models and evaluate once on the test set

In [6]:
test_rows = []
test_predictions = {}

for _, choice in best_by_feature_set.iterrows():
    h = int(choice["horizon_steps"])
    feature_name = choice["feature_set"]
    model_name = choice["model"]
    params = choice["params"]
    features = feature_sets[feature_name]

    d = data[h]
    train_val = d[d["SPLIT"].isin(["train", "val"])]
    test = d[d["SPLIT"] == "test"]

    model = make_model(model_name, params)
    model.fit(train_val[features], train_val["TARGET"])

    actual = test["TARGET"].to_numpy()
    predicted = np.clip(model.predict(test[features]), 0, None)

    daytime_mask = (
        (test["TARGET_TIME"].dt.hour >= 6)
        & (test["TARGET_TIME"].dt.hour < 18)
    ).to_numpy()

    for period, mask in [
        ("all_rows", np.ones(len(test), dtype=bool)),
        ("daytime", daytime_mask),
    ]:
        test_rows.append({
            "horizon_minutes": HORIZONS[h],
            "feature_set": feature_name,
            "model": model_name,
            "params": choice["params"],
            "period": period,
            "n_rows": int(mask.sum()),
            "MAE": mean_absolute_error(actual[mask], predicted[mask]),
            "RMSE": mean_squared_error(actual[mask], predicted[mask]) ** 0.5,
            "R2": r2_score(actual[mask], predicted[mask]),
        })

    test_predictions[(h, feature_name)] = pd.DataFrame({
        "TARGET_TIME": test["TARGET_TIME"].to_numpy(),
        "actual": actual,
        "predicted": predicted,
    }).reset_index(drop=True)

test_results = pd.DataFrame(test_rows)
display(test_results.round(3))

,horizon_minutes,feature_set,model,params,period,n_rows,MAE,RMSE,R2
0,15,A_time,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'n_est...",all_rows,473,1111.945,2137.923,0.933
1,15,A_time,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'n_est...",daytime,234,2128.267,3036.006,0.832
2,15,A_weather,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est...",all_rows,473,1176.296,2259.151,0.925
3,15,A_weather,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est...",daytime,234,2241.410,3205.526,0.813
4,15,B_history,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 3}",all_rows,473,1125.231,2301.868,0.922
5,15,B_history,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 3}",daytime,234,2252.711,3270.058,0.805
6,15,B_time,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'n_est...",all_rows,473,1123.510,2129.354,0.933
7,15,B_time,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'n_est...",daytime,234,2126.907,3023.286,0.834
8,30,A_time,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'n_est...",all_rows,471,1402.594,2613.898,0.900
9,30,A_time,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'n_est...",daytime,233,2703.016,3708.492,0.750


Calculate persistence and smart-persistence baselines

In [7]:
baseline_rows = []

for h, d in data.items():
    test = d[d["SPLIT"] == "test"].copy()

    power_lookup = d.set_index("DATE_TIME")["AC_POWER"]

    test["pred_persistence"] = test["AC_POWER"]

    yesterday_time = test["TARGET_TIME"] - pd.Timedelta(days=1)
    test["pred_yesterday"] = yesterday_time.map(power_lookup)

    train_val = d[d["SPLIT"].isin(["train", "val"])].copy()
    train_val["minute_of_day"] = (
        train_val["DATE_TIME"].dt.hour * 60
        + train_val["DATE_TIME"].dt.minute
    )

    envelope = train_val.groupby("minute_of_day")["AC_POWER"].quantile(0.95)

    current_minute = test["DATE_TIME"].dt.hour * 60 + test["DATE_TIME"].dt.minute
    target_minute = test["TARGET_TIME"].dt.hour * 60 + test["TARGET_TIME"].dt.minute

    current_envelope = current_minute.map(envelope)
    target_envelope = target_minute.map(envelope)

    clear_sky_index = (
        test["AC_POWER"] / current_envelope.replace(0, np.nan)
    ).clip(lower=0, upper=1.5)

    test["pred_smart"] = clear_sky_index * target_envelope

    daytime = (
        (test["TARGET_TIME"].dt.hour >= 6)
        & (test["TARGET_TIME"].dt.hour < 18)
    )

    for baseline_name, prediction_column in [
        ("Persistence", "pred_persistence"),
        ("Same time yesterday", "pred_yesterday"),
        ("Smart persistence", "pred_smart"),
    ]:
        for period, period_mask in [
            ("all_rows", pd.Series(True, index=test.index)),
            ("daytime", daytime),
        ]:
            valid = period_mask & test[prediction_column].notna()
            y_true = test.loc[valid, "TARGET"]
            y_pred = test.loc[valid, prediction_column]

            baseline_rows.append({
                "horizon_minutes": HORIZONS[h],
                "baseline": baseline_name,
                "period": period,
                "n_rows": len(y_true),
                "MAE": mean_absolute_error(y_true, y_pred),
                "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
                "R2": r2_score(y_true, y_pred),
            })

baseline_results = pd.DataFrame(baseline_rows)
display(baseline_results.round(3))

,horizon_minutes,baseline,period,n_rows,MAE,RMSE,R2
0,15,Persistence,all_rows,473,1206.363,2321.379,0.921
1,15,Persistence,daytime,234,2395.227,3295.760,0.802
2,15,Same time yesterday,all_rows,472,1980.157,3837.892,0.783
3,15,Same time yesterday,daytime,234,3983.165,5450.428,0.459
4,15,Smart persistence,all_rows,249,2148.869,3193.032,0.830
5,15,Smart persistence,daytime,229,2330.876,3329.368,0.789
6,30,Persistence,all_rows,471,1775.827,3156.084,0.854
7,30,Persistence,daytime,233,3486.617,4472.088,0.637
8,30,Same time yesterday,all_rows,469,1991.671,3850.066,0.783
9,30,Same time yesterday,daytime,233,3997.940,5461.997,0.458


Compare feature sets and models with baselines

In [8]:
comparisons = []

for period in ["all_rows", "daytime"]:
    for horizon in HORIZONS.values():
        rows = test_results[
            (test_results["horizon_minutes"] == horizon)
            & (test_results["period"] == period)
        ]
        values = rows.set_index("feature_set")["MAE"]

        comparisons.append({
            "horizon_minutes": horizon,
            "period": period,
            "A_weather_MAE": values.get("A_weather", np.nan),
            "B_history_MAE": values.get("B_history", np.nan),
            "B_minus_A_MAE": (
                values.get("B_history", np.nan)
                - values.get("A_weather", np.nan)
            ),
            "A_time_MAE": values.get("A_time", np.nan),
            "B_time_MAE": values.get("B_time", np.nan),
            "B_time_minus_A_time_MAE": (
                values.get("B_time", np.nan)
                - values.get("A_time", np.nan)
            ),
        })

research_comparison = pd.DataFrame(comparisons)

ml_comparison = test_results[
    ["horizon_minutes", "feature_set", "period", "n_rows", "MAE", "RMSE", "R2", "model"]
].rename(columns={"model": "method"})

baseline_comparison = baseline_results[
    ["horizon_minutes", "baseline", "period", "n_rows", "MAE", "RMSE", "R2"]
].rename(columns={"baseline": "method"})

baseline_comparison["feature_set"] = "baseline"

model_and_baseline = pd.concat(
    [ml_comparison, baseline_comparison],
    ignore_index=True,
)

display(research_comparison.round(3))
display(
    model_and_baseline
    .sort_values(["period", "horizon_minutes", "MAE"])
    .round(3)
)

,horizon_minutes,period,A_weather_MAE,B_history_MAE,B_minus_A_MAE,A_time_MAE,B_time_MAE,B_time_minus_A_time_MAE
0,15,all_rows,1176.296,1125.231,-51.065,1111.945,1123.510,11.565
1,30,all_rows,1630.770,1494.817,-135.953,1402.594,1385.892,-16.703
2,45,all_rows,1749.122,1780.854,31.732,1480.665,1506.537,25.871
3,60,all_rows,1859.746,1799.376,-60.369,1516.915,1636.007,119.092
4,15,daytime,2241.410,2252.711,11.301,2128.267,2126.907,-1.360
5,30,daytime,3035.845,2965.797,-70.048,2703.016,2639.568,-63.448
6,45,daytime,3394.374,3254.171,-140.203,2806.198,2815.622,9.424
7,60,daytime,3514.880,3419.036,-95.844,2907.837,3024.305,116.468


,horizon_minutes,feature_set,period,n_rows,MAE,RMSE,R2,method
0,15,A_time,all_rows,473,1111.945,2137.923,0.933,GradientBoosting
6,15,B_time,all_rows,473,1123.510,2129.354,0.933,GradientBoosting
4,15,B_history,all_rows,473,1125.231,2301.868,0.922,RandomForest
2,15,A_weather,all_rows,473,1176.296,2259.151,0.925,GradientBoosting
32,15,baseline,all_rows,473,1206.363,2321.379,0.921,Persistence
34,15,baseline,all_rows,472,1980.157,3837.892,0.783,Same time yesterday
36,15,baseline,all_rows,249,2148.869,3193.032,0.830,Smart persistence
14,30,B_time,all_rows,471,1385.892,2574.185,0.903,GradientBoosting
8,30,A_time,all_rows,471,1402.594,2613.898,0.900,GradientBoosting
12,30,B_history,all_rows,471,1494.817,2965.711,0.871,RandomForest


##Paired moving-block bootstrap for history comparisons
This resamples 16 consecutive rows within each test day. It compares paired absolute errors, so both models are evaluated on the same target timestamps.

In [9]:
rng = np.random.default_rng(42)
N_BOOTSTRAP = 2000
BLOCK_SIZE = 16

feature_pairs = [
    ("A_weather", "B_history"),
    ("A_time", "B_time"),
]

bootstrap_rows = []

def resample_day_blocks(values, block_size, rng):
    values = np.asarray(values)
    n = len(values)

    if n == 0:
        return values

    if n < block_size:
        return rng.choice(values, size=n, replace=True)

    sampled = []

    while len(sampled) < n:
        start = rng.integers(0, n - block_size + 1)
        sampled.extend(values[start:start + block_size])

    return np.asarray(sampled[:n])


for h in HORIZONS:
    for model_a, model_b in feature_pairs:
        a = test_predictions[(h, model_a)].rename(
            columns={"actual": "actual_a", "predicted": "pred_a"}
        )
        b = test_predictions[(h, model_b)].rename(
            columns={"actual": "actual_b", "predicted": "pred_b"}
        )

        paired = a.merge(
            b,
            on="TARGET_TIME",
            validate="one_to_one",
        )

        if not np.allclose(paired["actual_a"], paired["actual_b"]):
            raise ValueError(f"Targets do not match for horizon {h}.")

        paired["day"] = paired["TARGET_TIME"].dt.normalize()
        paired["error_a"] = (paired["actual_a"] - paired["pred_a"]).abs()
        paired["error_b"] = (paired["actual_b"] - paired["pred_b"]).abs()

        for period in ["all_rows", "daytime"]:
            sample = paired.copy()

            if period == "daytime":
                hour = sample["TARGET_TIME"].dt.hour
                sample = sample[(hour >= 6) & (hour < 18)]

            if sample.empty:
                continue

            sample["delta"] = sample["error_b"] - sample["error_a"]
            observed = sample["delta"].mean()
            day_groups = [
                group["delta"].to_numpy()
                for _, group in sample.groupby("day")
            ]

            bootstrap_estimates = []

            for _ in range(N_BOOTSTRAP):
                resampled_days = [
                    resample_day_blocks(group, BLOCK_SIZE, rng)
                    for group in day_groups
                ]
                bootstrap_estimates.append(
                    np.concatenate(resampled_days).mean()
                )

            low, high = np.percentile(bootstrap_estimates, [2.5, 97.5])

            bootstrap_rows.append({
                "horizon_minutes": HORIZONS[h],
                "comparison": f"{model_b} minus {model_a}",
                "period": period,
                "MAE_difference": observed,
                "CI_2.5%": low,
                "CI_97.5%": high,
            })

bootstrap_results = pd.DataFrame(bootstrap_rows)
display(bootstrap_results.round(3))

,horizon_minutes,comparison,period,MAE_difference,CI_2.5%,CI_97.5%
0,15,B_history minus A_weather,all_rows,-51.065,-96.394,13.424
1,15,B_history minus A_weather,daytime,11.301,-71.864,101.261
2,15,B_time minus A_time,all_rows,11.565,-18.298,31.416
3,15,B_time minus A_time,daytime,-1.360,-50.687,48.973
4,30,B_history minus A_weather,all_rows,-135.953,-216.155,-56.617
5,30,B_history minus A_weather,daytime,-70.048,-231.442,39.366
6,30,B_time minus A_time,all_rows,-16.703,-69.633,13.702
7,30,B_time minus A_time,daytime,-63.448,-155.382,-27.949
8,45,B_history minus A_weather,all_rows,31.732,-116.488,116.581
9,45,B_history minus A_weather,daytime,-140.203,-352.780,-66.047
